In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_keys = "cytokine"
condition_rep_keys = "condition_embeddings"
donor_rep_keys = "donor_one_hot"
random_seed = 42
dataset_name = "PBMC_donor11_hvg_e5test3"
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_state"
if_adata_ref = None
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

In [6]:
filePath = './data/raw/PBMC_donor11_hvg.h5ad'
adata = sc.read_h5ad(filePath)
print(adata)

AnnData object with n_obs × n_vars = 613365 × 5412
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'


In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == "PBS")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    569410
True      43955
Name: count, dtype: int64


In [8]:
#adata.obs[donor_rep_keys] = 7
print(adata.obs[donor_rep_keys].value_counts())

donor_one_hot
11    613365
Name: count, dtype: int64


## splitting

In [9]:
#adata = adata[~adata.obs[condition_keys].isin(["LT-alpha2-beta1","IFN-lambda2","IFN-lambda3","IL-18Ra","LT-alpha1-beta2"])]

In [10]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 / 6
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 / 6

['PRL', 'TRAIL', 'EGF', 'CD27L', 'IL-32-beta', 'IL-6', 'IL-11', 'SCF', 'IL-17F', 'OSM', 'IL-19', 'CD30L', 'IL-36Ra', 'IL-21', 'IL-7', 'G-CSF', 'IL-3', 'LIF']
['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'Megalin', 'C5a', 'CD40L', 'CT-1', 'Decorin', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'GDNF', 'GITRL', 'GM-CSF', 'HGF', 'IFN-alpha1', 'IFN-beta', 'IFN-epsilon', 'IFN-gamma', 'IFN-lambda1', 'IFN-lambda2', 'IFN-lambda3', 'IFN-omega', 'IGF-1', 'IL-1-alpha', 'IL-1-beta', 'IL-10', 'IL-12', 'IL-13', 'IL-15', 'IL-16', 'IL-17A', 'IL-17B', 'IL-17C', 'IL-17D', 'IL-17E', 'IL-18Ra', 'IL-1Ra', 'IL-2', 'IL-20', 'IL-22', 'IL-23', 'IL-24', 'IL-26', 'IL-27', 'IL-31', 'IL-33', 'IL-34', 'IL-35', 'IL-36-alpha', 'IL-4', 'IL-5', 'IL-8', 'IL-9', 'LIGHT', 'LT-alpha1-beta2', 'LT-alpha2-beta1', 'Leptin', 'M-CSF', 'Noggin', 'OX40L', 'PSPN', 'RANKL', 'LAP-TGF-beta1', 'TL1A', 'TNF-alpha', 'TPO', 'TSLP', 'TWEAK', 'VEGF']


In [11]:
del adata

## latent embedding

In [12]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
condition_rep_dict = pd.read_pickle("./data/processed/condition_embedding_e5_test5.pkl")
condition_rep_dict = {
    k: (v["embedding"] if isinstance(v, dict) and "embedding" in v else None)
    for k, v in condition_rep_dict.items()
}
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [13]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [14]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    batch_key = "donor",
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep #+ "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

[12.19878     6.3304415   5.108692    2.788115    2.6199658   2.5855775
  2.466213    2.1122458   1.8650693   1.7538075   1.6688269   1.6707683
  1.4710318   1.5381949   1.4024265   1.3934157   1.3290818   1.3106625
  1.3176571   1.2737206   1.2879856   1.2281953   1.2477257   1.2116311
  1.1922541   1.1862013   1.1864771   1.1552886   1.1449716   1.141044
  1.1373132   1.1237748   1.1162826   1.0862373   1.095503    1.0941029
  1.0720576   1.0640043   1.0655527   1.0481948   1.0491517   1.0509894
  1.0469654   1.0373244   1.0407214   1.035317    1.0163375   1.0278934
  1.0086732   1.0209459   1.0106643   1.0031437   0.99849355  1.0024866
  0.99832004  0.9911125   1.0007852   0.9847058   0.9822227   0.98391336
  0.9719225   0.97304755  0.96855175  0.9736751   0.9666621   0.9552178
  0.96183217  0.9626792   0.95297086  0.9542847   0.9537249   0.9467323
  0.9589987   0.94115824  0.94837916  0.9452242   0.93591934  0.9351141
  0.9317229   0.93101066  0.93857557  0.93238705  0.92372906  0.

In [15]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [16]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/PBMC_donor11_hvg_e5test5_42_0.2_True_X_pca_100_None
AnnData object with n_obs × n_vars = 43955 × 5412
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'donor_one_hot', 'is_control'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'normalized_m', 'pca'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 455637 × 5412
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'd

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()